# Fine-tuning DziriBERT — Sentiment Derja (Production)

**Pipeline complet** : Google Drive → préparation → entraînement → évaluation → sauvegarde Drive

| Élément | Détail |
|---------|--------|
| **Modèle** | `alger-ia/dziribert` (BERT pré-entraîné Derja) |
| **Dataset** | 1744 commentaires Facebook (Ramy + Hamoud Boualem) |
| **Classes** | 3 — positive, negative, neutral |
| **Stratégie** | CrossEntropy pondérée (class weights inverse frequency) |
| **Seuil MVP** | F1 macro ≥ 0.70, F1 par classe ≥ 0.60 |

> Le modèle final tourne en production **sans aucune dépendance LLM**.

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import torch, json, os, time
import numpy as np
import pandas as pd
import torch.nn as nn
from collections import Counter
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU : {gpu} ({vram:.1f} GB)")
else:
    print("⚠️ Pas de GPU — va dans Runtime → Change runtime type → T4")

## 2. Charger les données depuis Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

JSON_PATH = "/content/drive/MyDrive/annotated_comments.json"
assert os.path.exists(JSON_PATH), f"Fichier non trouvé : {JSON_PATH}"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Chargé : {len(raw_data)} commentaires")

## 3. Préparer le dataset 3 classes

La classe `mixed` (34 exemples, trop peu pour être apprise) est fusionnée
dans la classe la plus probable selon le label dur argmax des soft labels.

In [ ]:
LABEL2ID = {"positive": 0, "negative": 1, "neutral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3
SEED = 42

# ── Fusion mixed ──
records = []
for item in raw_data:
    cands = item["candidates"]
    scores = {"positive": cands["pos"], "negative": cands["neg"], "neutral": cands["neu"]}

    if item["sentiment"] == "mixed":
        label = max(scores, key=scores.get)
    else:
        label = item["sentiment"]

    if label not in LABEL2ID:
        continue

    records.append({"text": item["text"], "label": label, "label_id": LABEL2ID[label]})

df = pd.DataFrame(records)
print(f"Total : {len(df)}")
print(df["label"].value_counts())

## 4. Class weights + Split stratifié

In [ ]:
from sklearn.model_selection import train_test_split

# ── Class weights (inverse frequency) ──
counts = Counter(df["label"])
n = len(df)
class_weights = {lab: n / (NUM_LABELS * c) for lab, c in counts.items()}
weights_tensor = torch.tensor(
    [class_weights[ID2LABEL[i]] for i in range(NUM_LABELS)],
    dtype=torch.float32,
).to(device)

print("Class weights :")
for lab in LABEL2ID:
    print(f"  {lab:10s} : {class_weights[lab]:.4f}")

# ── Split 70/15/15 stratifié ──
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

for name, s in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name:5s} : {len(s):4d}  {dict(Counter(s['label']))}")

## 5. Tokenisation

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "alger-ia/dziribert"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

from torch.utils.data import Dataset, DataLoader

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(texts, padding="max_length", truncation=True,
                                   max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

train_ds = CommentDataset(train_df["text"].tolist(), train_df["label_id"].tolist(), tokenizer, MAX_LEN)
val_ds   = CommentDataset(val_df["text"].tolist(), val_df["label_id"].tolist(), tokenizer, MAX_LEN)
test_ds  = CommentDataset(test_df["text"].tolist(), test_df["label_id"].tolist(), tokenizer, MAX_LEN)

print(f"Datasets — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

## 6. Modèle + Trainer pondéré

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
)
model.to(device)
print(f"Params : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class WeightedTrainer(Trainer):
    """CrossEntropy avec class weights pour compenser le déséquilibre."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=weights_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per = f1_score(labels, preds, average=None)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_positive": f1_per[0],
        "f1_negative": f1_per[1],
        "f1_neutral": f1_per[2],
        "accuracy": accuracy_score(labels, preds),
    }

## 7. Hyper-paramètres

| Param | Valeur | Raison |
|-------|--------|--------|
| LR | 2e-5 | Standard BERT fine-tuning |
| Epochs | 10 | Early stopping coupe avant |
| Batch | 16 | Optimal pour T4 (15 GB) |
| Warmup | 10% | Stabilise le début |
| Patience | 3 | Stop si F1 stagne 3 epochs |

In [ ]:
EPOCHS = 10
BATCH = 16
LR = 2e-5
PATIENCE = 3

args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    resume_from_checkpoint=None,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

## 8. Entraînement

In [ ]:
result = trainer.train()

print(f"\nTemps : {result.metrics['train_runtime']:.0f}s")
print(f"Loss  : {result.metrics['train_loss']:.4f}")

## 9. Évaluation sur le Test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_out = trainer.predict(test_ds)
y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = test_out.label_ids
metrics = test_out.metrics

label_names = ["positive", "negative", "neutral"]

print("=" * 55)
print("  RÉSULTATS — TEST SET")
print("=" * 55)
print(f"  F1 macro    : {metrics['test_f1_macro']:.4f}")
print(f"  F1 positive : {metrics['test_f1_positive']:.4f}")
print(f"  F1 negative : {metrics['test_f1_negative']:.4f}")
print(f"  F1 neutral  : {metrics['test_f1_neutral']:.4f}")
print(f"  Accuracy    : {metrics['test_accuracy']:.4f}")

print("\n  SEUILS MVP :")
for name, val, thresh in [
    ("F1 macro", metrics["test_f1_macro"], 0.70),
    ("F1 positive", metrics["test_f1_positive"], 0.60),
    ("F1 negative", metrics["test_f1_negative"], 0.60),
    ("F1 neutral", metrics["test_f1_neutral"], 0.60),
]:
    ok = "✅" if val >= thresh else "❌"
    print(f"  {ok} {name:15s} {val:.4f} (seuil {thresh})")

print("\n" + classification_report(y_true, y_pred, target_names=label_names, digits=4))

## 10. Matrice de confusion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_xlabel("Prédit"); axes[0].set_ylabel("Réel")
axes[0].set_title("Matrice de confusion")

f1s = [metrics["test_f1_positive"], metrics["test_f1_negative"], metrics["test_f1_neutral"]]
bars = axes[1].bar(label_names, f1s, color=["#2ecc71","#e74c3c","#95a5a6"], edgecolor="k", lw=0.5)
axes[1].axhline(0.60, color="orange", ls="--", label="Seuil (0.60)")
axes[1].axhline(0.70, color="red", ls="--", label="Cible (0.70)")
axes[1].set_ylim(0, 1); axes[1].set_ylabel("F1"); axes[1].legend()
axes[1].set_title("F1 par classe")
for b, s in zip(bars, f1s):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f"{s:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("/content/results.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Sauvegarder le modèle dans Google Drive

In [ ]:
import shutil

DRIVE_DIR = "/content/drive/MyDrive/RamyPulse/models/dziribert-sentiment"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Modèle + tokenizer
trainer.save_model(DRIVE_DIR)
tokenizer.save_pretrained(DRIVE_DIR)

# Métadonnées
meta = {
    "model_base": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "class_weights": {k: round(v, 4) for k, v in class_weights.items()},
    "dataset": {"total": len(df), "train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
    "test_metrics": {k.replace("test_", ""): round(v, 4) for k, v in metrics.items() if k.startswith("test_")},
    "hyperparams": {"lr": LR, "batch": BATCH, "epochs_max": EPOCHS, "patience": PATIENCE, "max_len": MAX_LEN},
}
with open(f"{DRIVE_DIR}/training_results.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

shutil.copy("/content/results.png", f"{DRIVE_DIR}/results.png")

print(f"Modèle sauvegardé : {DRIVE_DIR}/")
for fn in sorted(os.listdir(DRIVE_DIR)):
    print(f"  {fn:40s} {os.path.getsize(f'{DRIVE_DIR}/{fn}')/1024:.0f} KB")

## 12. Test d'inférence

In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model=DRIVE_DIR, tokenizer=DRIVE_DIR,
               device=0 if torch.cuda.is_available() else -1, top_k=None)

exemples = [
    ("رامي عصير ممتاز والله نحبو بزاف", "positive"),
    ("Ramy jus top qualité", "positive"),
    ("حمود بوعلام ديما الأفضل", "positive"),
    ("المنتوج هذا ماشي مليح خلاص", "negative"),
    ("had el produit dégoûtant wallah", "negative"),
    ("لقاوها تسقي بمياه الصرف الصحي", "negative"),
    ("شحال الثمن تع رامي الجديد؟", "neutral"),
    ("win nlgah ramy f béjaia?", "neutral"),
    ("أنا أشارك شهر مارس", "neutral"),
]

print("=" * 65)
print("  TEST D'INFÉRENCE — DziriBERT Sentiment")
print("=" * 65)
ok = 0
for text, expected in exemples:
    res = sorted(clf(text)[0], key=lambda x: x["score"], reverse=True)
    pred = res[0]["label"]
    dist = " ".join(f"{r['label'][:3]}:{r['score']:.2f}" for r in res)
    match = "✅" if pred == expected else "❌"
    ok += pred == expected
    print(f"  {match} {pred:8s} | {dist} | {text[:50]}")
print(f"\n  {ok}/{len(exemples)} corrects")

## 13. Et après ?

**Si les seuils MVP sont atteints :**
- Le modèle dans `Drive/RamyPulse/models/dziribert-sentiment/` est prêt pour la production
- L'intégrer dans RamyPulse (`core/analysis/sentiment_classifier.py`)

**Si F1 macro < 0.70 :**
1. Baisser le LR à `1e-5`
2. Augmenter `EPOCHS` à 15 et `PATIENCE` à 5
3. Essayer Focal Loss (remplacer `nn.CrossEntropyLoss` par `FocalLoss` — voir `distillation/finetune_dziribert.ipynb`)
4. Si toujours insuffisant → essayer CAMeLBERT ou MarBERT comme base